# Homework Set 2
# Predicting Bandgaps for Water-Splitting Material Discovery

**EMA 4935 / EMA 6938 — AI for Materials, Spring 2026**

---

### Overview

Photoelectrochemical water splitting is a promising route to sustainable hydrogen production. The key material requirement is a semiconductor whose **bandgap** is large enough to drive the water-splitting reaction (thermodynamic minimum $E_g \geq 1.23$ eV) yet not so large that most of the solar spectrum is wasted. In practice, overpotentials and kinetic losses mean that a **bandgap of roughly 1.5–2.5 eV** is considered optimal.

In this homework you will:

1. Explore a real dataset of DFT-computed bandgaps from the **Materials Project**.
2. Engineer features from chemical composition (Magpie descriptors, oxidation-state statistics) and crystal structure (density, symmetry, etc.).
3. Train and compare **two regression models** of your choice to predict the bandgap.
4. Apply your best model to a **held-out set of ~200 candidate materials** and identify the most promising water-splitting candidates.

---

### What You Will Need

| Package | Purpose |
|---|---|
| `pandas`, `numpy` | Data handling |
| `matminer` | Composition-based featurizers (Magpie, OxidationStates) |
| `pymatgen` | Composition objects |
| `scikit-learn` | Regression models, cross-validation, metrics |
| `matplotlib`, `seaborn` | Visualization |

Install any missing packages with:
```bash
pip install matminer pymatgen scikit-learn
```

---

### Data Files Provided

| File | Description |
|---|---|
| `bandgap_training.csv` | Training set (~4 800 materials) with columns described below |
| `bandgap_candidates.csv` | Held-out screening set (~200 materials); **bandgap column is hidden** |

**Column descriptions** (training set):

| Column | Type | Description |
|---|---|---|
| `mp_id` | str | Materials Project identifier (e.g., mp-149) |
| `formula` | str | Reduced chemical formula |
| `bandgap_eV` | float | PBE bandgap in eV (target variable) |
| `nsites` | int | Number of atoms in the unit cell |
| `density_g_per_cm3` | float | Crystal density (g/cm³) |
| `volume_per_atom_A3` | float | Unit-cell volume per atom (Å³) |
| `spacegroup_number` | int | International spacegroup number (1–230) |
| `crystal_system` | str | Crystal system (cubic, hexagonal, …) |
| `nelements` | int | Number of distinct elements |

The candidate file has the same columns **except** `bandgap_eV`, which you will predict.

---

### Grading Breakdown

| Part | Points |
|---|---|
| Part 1 — Data Exploration | 15 |
| Part 2 — Feature Engineering | 30 |
| Part 3 — Regression Modeling | 35 |
| Part 4 — Water-Splitting Screening | 20 |
| **Total** | **100** |

---
---

## Part 1. Data Loading and Exploration (15 points)

### Tasks

1. Load `bandgap_training.csv` into a pandas DataFrame.
2. Report the shape of the dataset and display the first few rows.
3. Compute and report summary statistics for `bandgap_eV` (mean, median, standard deviation, min, max).
4. Create the following visualizations:
   - A **histogram** of bandgap values. Comment on the distribution shape.
   - A **bar chart** showing the number of materials per crystal system.
   - A **scatter plot** of `density_g_per_cm3` vs. `bandgap_eV`, colored by crystal system.
5. Briefly discuss (2–3 sentences): Do you observe any obvious relationship between the structural properties and the bandgap? Are there any data quality issues you notice?

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Load the training data
df_train = pd.read_csv('bandgap_training.csv')

# 2. Shape and preview
print(f"Training set shape: {df_train.shape}")
df_train.head()

In [ ]:
# 3. Summary statistics for bandgap_eV
# TODO: compute and print mean, median, std, min, max


In [ ]:
# 4a. Histogram of bandgap values
# TODO: create a histogram with appropriate bins and labels

# 4b. Bar chart: materials per crystal system
# TODO: create a bar chart

# 4c. Scatter plot: density vs bandgap, colored by crystal system
# TODO: create a scatter plot


**Discussion (Task 5):**

*Your answer here.*

---
---

## Part 2. Feature Engineering (30 points)

Good features are the foundation of any ML model for materials. You will create three groups of features and combine them into a single feature matrix.

### Background: matminer Featurizers

[matminer](https://hackingmaterials.lbl.gov/matminer/) is a Python library for data mining in materials science. It provides **featurizers** — classes that take a material representation (composition, structure, etc.) and output a vector of numerical descriptors. Each featurizer has a `featurize_dataframe()` method that adds new columns directly to a DataFrame.

In this homework you will use two composition-based featurizers:

- **Magpie descriptors** (`ElementProperty` with preset `"magpie"`): Computes statistics (mean, std, min, max, range, mode) of elemental properties such as atomic number, electronegativity, row/column in the periodic table, atomic radius, and many more. This yields a rich set of ~130 features per material.

- **Oxidation-state statistics** (`OxidationStates`): Estimates likely oxidation states for each element in the composition and computes statistics of those oxidation states.

You will also use the **structural features** already present in the CSV (density, volume per atom, spacegroup number, number of sites).

---

### Tasks

1. **Create a `Composition` column** by converting the `formula` strings into `pymatgen.core.Composition` objects. This is required by the matminer featurizers.

2. **Generate Magpie features** using matminer's `ElementProperty` featurizer with the `"magpie"` preset. Apply it to the training DataFrame.

3. **Generate oxidation-state features** using matminer's `OxidationStates` featurizer. Apply it to the training DataFrame.

4. **Assemble the full feature matrix** by combining:
   - The Magpie features (from Task 2)
   - The oxidation-state features (from Task 3)
   - The structural features already in the CSV: `nsites`, `density_g_per_cm3`, `volume_per_atom_A3`, `spacegroup_number`, `nelements`
   - One-hot encode `crystal_system` (use `pd.get_dummies`)

5. **Handle missing values**: Some featurizers may produce NaN for certain compositions. Report how many rows contain NaN values, then choose a strategy (drop rows or impute) and justify your choice.

6. **Report** the final feature matrix dimensions and list the top 10 features most correlated with `bandgap_eV` (by absolute Pearson correlation).

---

In [ ]:
from pymatgen.core import Composition

# Task 1: Create Composition objects from formula strings
# TODO: Add a 'composition' column to df_train
df_train['composition'] = df_train['formula'].apply(Composition)

print(f"Example composition: {df_train['composition'].iloc[0]}")
print(f"Type: {type(df_train['composition'].iloc[0])}")

In [ ]:
from matminer.featurizers.composition import ElementProperty

# Task 2: Generate Magpie features
magpie_featurizer = ElementProperty.from_preset("magpie")

# TODO: Apply the featurizer to df_train using featurize_dataframe()
# Hint: magpie_featurizer.featurize_dataframe(df_train, col_id='composition')
# Note: This may take a minute or two to run.


In [ ]:
from matminer.featurizers.composition import OxidationStates

# Task 3: Generate oxidation-state features
# TODO: Create the OxidationStates featurizer and apply it to df_train


In [ ]:
# Task 4: Assemble the full feature matrix

# Structural features from the CSV
structural_cols = ['nsites', 'density_g_per_cm3', 'volume_per_atom_A3',
                   'spacegroup_number', 'nelements']

# TODO: One-hot encode crystal_system
# TODO: Identify the Magpie and OxidationStates column names
# TODO: Combine all features into a single DataFrame X
# TODO: Define the target vector y = df_train['bandgap_eV']


In [ ]:
# Task 5: Handle missing values
# TODO: Report the number and fraction of rows with any NaN
# TODO: Choose and apply your strategy (drop or impute)
# TODO: Justify your choice in a comment or markdown cell below


In [ ]:
# Task 6: Report feature matrix dimensions and top-10 correlated features
# TODO: Print X.shape
# TODO: Compute absolute Pearson correlation of each feature with y
# TODO: Print the top 10


---
---

## Part 3. Regression Modeling (35 points)

### Tasks

1. **Choose two regression methods.** You may pick from (but are not limited to):
   - Ridge / Lasso regression
   - Support Vector Regression (SVR)
   - Random Forest Regression
   - Gradient Boosted Trees (e.g., `GradientBoostingRegressor` or `XGBRegressor`)
   - Kernel Ridge Regression
   - k-Nearest Neighbors Regression
   - Neural Network (`MLPRegressor`)

   **Write 3–4 sentences justifying your choice of each method.** Consider factors such as:
   - How does the method handle high-dimensional feature spaces?
   - Is it robust to correlated or irrelevant features?
   - Does it make assumptions about the functional form (linear vs. nonlinear)?
   - Computational cost and scalability.

2. **Split the data** into training (80%) and test (20%) sets using `train_test_split` with `random_state=42`.

3. **Hyperparameter tuning**: For each model, select at least **2 hyperparameters** to tune. Use **5-fold cross-validation** on the training set (e.g., `GridSearchCV` or `RandomizedSearchCV`) to find good hyperparameter values.
   - Report the hyperparameters you searched over and the values you tried.
   - Report the best hyperparameters found.

4. **Evaluate both models** on the held-out test set using:
   - Mean Absolute Error (MAE)
   - Root Mean Squared Error (RMSE)
   - Coefficient of determination ($R^2$)

5. **Parity plot**: For each model, create a scatter plot of predicted vs. actual bandgap on the test set. Include the identity line ($y = x$) and annotate the plot with the MAE and $R^2$.

6. **Learning curves**: For your best model, plot training and validation MAE as a function of training set size (use `sklearn.model_selection.learning_curve`). Discuss what the learning curves tell you about bias and variance.

7. **Compare and discuss**: Which model performed better and why? Would you expect these results to generalize to materials not in the Materials Project? What are the limitations of using PBE bandgaps as training targets?

---

### Task 1: Model Selection Justification

**Model 1:** *[Name your first model]*

*Your justification here (3–4 sentences).*

**Model 2:** *[Name your second model]*

*Your justification here (3–4 sentences).*

---

In [ ]:
from sklearn.model_selection import train_test_split

# Task 2: Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training set: {X_train.shape[0]} samples, {X_train.shape[1]} features")
print(f"Test set:     {X_test.shape[0]} samples")

In [ ]:
from sklearn.model_selection import GridSearchCV, cross_val_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Task 3: Model 1 — Hyperparameter tuning
# TODO: Import your chosen model
# TODO: Define the parameter grid (at least 2 hyperparameters)
# TODO: Run GridSearchCV with 5-fold CV
# TODO: Print best parameters and best CV score

# Example skeleton (replace with your model):
# from sklearn.ensemble import RandomForestRegressor
# param_grid_1 = {
#     'n_estimators': [100, 300, 500],
#     'max_depth': [10, 20, None],
# }
# model_1 = GridSearchCV(RandomForestRegressor(random_state=42),
#                        param_grid_1, cv=5, scoring='neg_mean_absolute_error',
#                        n_jobs=-1, verbose=1)
# model_1.fit(X_train, y_train)
# print(f"Best params: {model_1.best_params_}")
# print(f"Best CV MAE: {-model_1.best_score_:.3f} eV")


In [ ]:
# Task 3: Model 2 — Hyperparameter tuning
# TODO: Import your chosen model
# TODO: Define the parameter grid (at least 2 hyperparameters)
# TODO: Run GridSearchCV with 5-fold CV
# TODO: Print best parameters and best CV score


In [ ]:
# Task 4: Evaluate both models on the test set
# TODO: Predict on X_test for both models
# TODO: Compute MAE, RMSE, R² for both
# TODO: Print a comparison table

# Skeleton:
# y_pred_1 = model_1.predict(X_test)
# y_pred_2 = model_2.predict(X_test)
#
# results = pd.DataFrame({
#     'Model': ['Model 1 Name', 'Model 2 Name'],
#     'MAE (eV)': [mean_absolute_error(y_test, y_pred_1),
#                  mean_absolute_error(y_test, y_pred_2)],
#     'RMSE (eV)': [np.sqrt(mean_squared_error(y_test, y_pred_1)),
#                   np.sqrt(mean_squared_error(y_test, y_pred_2))],
#     'R²': [r2_score(y_test, y_pred_1),
#            r2_score(y_test, y_pred_2)],
# })
# print(results.to_string(index=False))


In [ ]:
# Task 5: Parity plots (predicted vs. actual)
# TODO: Create a 1x2 figure with a parity plot for each model
# TODO: Include the y=x line, and annotate with MAE and R²

# Skeleton:
# fig, axes = plt.subplots(1, 2, figsize=(12, 5))
# for ax, y_pred, name in zip(axes, [y_pred_1, y_pred_2],
#                              ['Model 1 Name', 'Model 2 Name']):
#     ax.scatter(y_test, y_pred, alpha=0.4, s=10)
#     lims = [0, max(y_test.max(), y_pred.max()) + 0.5]
#     ax.plot(lims, lims, 'k--', linewidth=1)
#     ax.set_xlabel('Actual Bandgap (eV)')
#     ax.set_ylabel('Predicted Bandgap (eV)')
#     ax.set_title(name)
#     mae = mean_absolute_error(y_test, y_pred)
#     r2 = r2_score(y_test, y_pred)
#     ax.annotate(f'MAE = {mae:.3f} eV\nR² = {r2:.3f}',
#                 xy=(0.05, 0.90), xycoords='axes fraction', fontsize=10)
# plt.tight_layout()
# plt.show()


In [ ]:
# Task 6: Learning curves for the best model
from sklearn.model_selection import learning_curve

# TODO: Generate learning curves
# TODO: Plot training and validation MAE vs. training set size
# Hint: Use scoring='neg_mean_absolute_error' and negate the scores for plotting


### Task 7: Discussion

*Your comparison and discussion here. Address:*
- *Which model performed better and why?*
- *What do the learning curves tell you about bias/variance?*
- *Would these results generalize to materials not in Materials Project?*
- *What are the limitations of using PBE bandgaps as training data?*

---
---

## Part 4. Water-Splitting Material Screening (20 points)

### Background

The overall water-splitting reaction is:

$$2\text{H}_2\text{O} \rightarrow 2\text{H}_2 + \text{O}_2, \qquad \Delta G^\circ = 237 \text{ kJ/mol}$$

This corresponds to a thermodynamic potential of **1.23 V** per electron. However, real devices require additional driving force to overcome overpotentials at the electrodes and resistive losses. A widely cited practical target is:

$$1.5 \text{ eV} \leq E_g \leq 2.5 \text{ eV}$$

Materials with bandgaps in this range can absorb a significant portion of the solar spectrum while providing enough thermodynamic driving force for the reaction.

### Tasks

1. **Load and featurize the candidate set**: Read `bandgap_candidates.csv` and generate the same features you used for training (Magpie, oxidation states, structural). Make sure the feature columns are in the **exact same order** as the training set.

2. **Predict bandgaps**: Use your best model to predict the bandgap for all ~200 candidate materials.

3. **Identify water-splitting candidates**: Filter the predictions to find materials with predicted bandgap in the range **1.5–2.5 eV**.

4. **Rank and report**: Rank the candidates by how close their predicted bandgap is to the ideal value of **2.0 eV** (i.e., sort by $|E_g^{\text{pred}} - 2.0|$). Display the top 10 candidates in a table with columns: `mp_id`, `formula`, `predicted_bandgap_eV`, `crystal_system`.

5. **Uncertainty discussion** (3–4 sentences): Your model has a finite test-set MAE. How should this prediction error inform your confidence in the screening results? If you were advising an experimentalist, how would you present these candidates?

---

In [ ]:
# Task 1: Load and featurize the candidate set
df_candidates = pd.read_csv('bandgap_candidates.csv')
print(f"Candidate set shape: {df_candidates.shape}")
df_candidates.head()

# TODO: Apply the SAME featurization pipeline you used in Part 2:
#   - Create Composition objects
#   - Apply Magpie featurizer
#   - Apply OxidationStates featurizer
#   - One-hot encode crystal_system
#   - Assemble features in the same column order as the training set
#
# IMPORTANT: The candidate feature matrix must have the exact same columns
# as the training feature matrix X. Use X.columns to verify alignment.


In [ ]:
# Task 2: Predict bandgaps for the candidate materials
# TODO: Use your best model to predict
# TODO: Add the predictions as a column to df_candidates


In [ ]:
# Task 3: Filter for water-splitting bandgap range (1.5 – 2.5 eV)
# TODO: Filter candidates
# TODO: Print how many candidates fall in the target range


In [ ]:
# Task 4: Rank by closeness to 2.0 eV and display top 10
# TODO: Sort by |predicted_bandgap - 2.0|
# TODO: Display top-10 table with mp_id, formula, predicted_bandgap_eV, crystal_system


### Task 5: Uncertainty Discussion

*Your answer here (3–4 sentences). Address:*
- *How does the model's test-set MAE affect your confidence in these predictions?*
- *How would you present these results to an experimentalist?*

---
---

## Submission Instructions

Submit your completed Jupyter Notebook (`.ipynb`) with:
- All code cells executed and producing output.
- All discussion questions answered in the provided markdown cells.
- Clear, well-commented code.

Ensure your notebook runs top-to-bottom without errors (Kernel → Restart & Run All).

---